In [0]:
# ===========================================================================
# Notebook  : 02a_silver_user
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02a_silver_user
# Purpose   : bronze.{customer_id}.crm_users → hgi.silver.users
# NOTE: User table provides owner/creator enrichment
# Serverless : Works on 2X-Small (safe_spark_conf skips unsupported configs)
# Job Compute: Full perf configs applied automatically
# ===========================================================================


In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/FINAL_pipeline_config.py

In [0]:
%run ./FINAL_silver_cdf_common.py

In [0]:
# CELL 2 ── Widgets
dbutils.widgets.text("customer_id",      "cust_001")
dbutils.widgets.text("starting_version", "0")
# starting_version='0' → replays full bronze history on first run
# After first run, checkpoint takes over (widget ignored)
# Set to a specific version number to skip already-processed history

customer_id      = dbutils.widgets.get("customer_id").strip().lower()
starting_version = dbutils.widgets.get("starting_version").strip()

In [0]:
# CELL 3 ── Object-specific constants
source_system = "salesforce"
object_name   = "user"


In [0]:
# CELL 4 ── Path and table resolution via pipeline_config helpers
bronze_full  = bronze_table(customer_id, object_name)
# bronze_full = bronze.{customer_id}.crm_users

target_full  = silver_table(object_name)
# target_full = hgi.silver.users

chk_path = checkpoint_path("silver", "salesforce", customer_id, "user")

print(f"=== 02a Silver CDF: SALESFORCE User ===")
print(f"  Customer         : {customer_id}")
print(f"  Bronze source    : {bronze_full}")
print(f"  Silver target    : {target_full}")
print(f"  Checkpoint       : {chk_path}")
print(f"  Starting version : {starting_version}")

In [0]:
# CELL 5 ── Create silver schema + table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")
create_silver_table(target_full, "users")
# uses SILVER_DDL_COLUMNS['users'] from pipeline_config

In [0]:
# CELL 6 ── Enable CDF on bronze source table
enable_cdf(bronze_full)

In [0]:
# CELL 7 ── Run CDF stream
run_cdf_stream(bronze_full, target_full, "salesforce", "user",
               chk_path, starting_version)

In [0]:
# CELL 8 ── Post-load OPTIMIZE
spark.sql(f"OPTIMIZE {target_full}")
print(f"Silver CDF complete: {target_full}")
dbutils.notebook.exit("Success")